# MVP — Análise de Dados e Boas Práticas

**Movimentação de óleo diesel A S10 em terminais aquaviários do Sudeste, 2025**

MBA PUC-Rio · Versão de trabalho · Validação inicial da base

# 1. Problema e perguntas

O objetivo é descrever os fluxos de recepção e entrega de diesel A S10 informados pelos terminais do Sudeste em 2025. A unidade de observação é um registro mensal de volume para uma combinação de terminal, produto, sentido, tipo de operação e modo de transporte. A unicidade dessa combinação será verificada na análise exploratória.

- **H1.** Como o volume mensal de recepção aquaviária de diesel A S10 se distribuiu entre os terminais do Sudeste em 2025?
- **H2.** Qual foi a participação de cada modo de transporte nas entregas de diesel A S10 registradas, por terminal com entregas reportadas?

A análise é descritiva e mantém recepções e entregas separadas. A base reserva é a série mensal de vendas de derivados e etanol da ANP.

# 2. Fonte e significado dos dados

[Movimentação dos Terminais Aquaviários — ANP](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/movimentacao-dos-terminais-aquaviarios) · [Dicionário oficial](https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/arquivos/arquivos-movimentacao-dos-terminais-aquaviarios/metadados-movimentacao-terminais-aquaviarios.pdf)

A carga usa o arquivo público da ANP. A fonte pode ser atualizada. O código registra o SHA-256 para identificar os bytes usados em cada execução. A política de redistribuição e o procedimento de fixação da versão serão verificados antes da entrega.

| Variável | Significado | Tipo / unidade |
|---|---|---|
| mes_de_referencia | Mês de referência | Data mensal |
| codigo_anp_do_terminal | Identificador do estabelecimento na ANP | Código |
| uf | UF do terminal | Categoria |
| sentido_da_operacao | Recepção (1) ou entrega (2) | Código |
| tipo_da_operacao | Categoria da operação, conforme dicionário | Código |
| modo_de_transporte | Rodoviário (1), ferroviário (2), aquaviário (4), dutoviário (5) ou outros (9) | Código |
| codigo_anp_do_produto | Identificador do produto | Código |
| descricao_do_produto | Descrição informada do produto | Texto |
| volume_m3 | Volume informado a 20 °C | m³ |

# 3. Ambiente e carregamento

Execute as células em ordem com Python, pandas e openpyxl. A fonte é baixada sem autenticação. O formato é identificado pelos bytes do arquivo, pois a extensão publicada pode divergir do conteúdo.

In [1]:
from hashlib import sha256
from importlib.metadata import version
from io import BytesIO
import sys
from urllib.request import urlopen

import pandas as pd

URL_ANP = (
    "https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-abertos/"
    "arquivos/arquivos-movimentacao-dos-terminais-aquaviarios/"
    "dados-abertos-movimentacao-terminais-aquaviarios.csv"
)

print("Python", sys.version.split()[0])
for pacote in ("pandas", "openpyxl"):
    print(pacote, version(pacote))

with urlopen(URL_ANP, timeout=60) as resposta:
    arquivo = resposta.read()

print("SHA-256", sha256(arquivo).hexdigest())

if arquivo.startswith(b"PK\x03\x04"):
    dados_origem = pd.read_excel(BytesIO(arquivo), engine="openpyxl")
    print("Formato recebido: Excel")
else:
    dados_origem = pd.read_csv(
        BytesIO(arquivo), sep=";", decimal=",", encoding="utf-8-sig"
    )
    print("Formato recebido: CSV")

dados = dados_origem.rename(
    columns=lambda nome: nome.strip().lower()
).rename(columns={"mesreferencia": "mes_de_referencia"})

colunas_necessarias = {
    "mes_de_referencia", "codigo_anp_do_terminal", "uf",
    "sentido_da_operacao", "tipo_da_operacao", "modo_de_transporte",
    "codigo_anp_do_produto", "descricao_do_produto", "volume_m3",
}
faltantes = colunas_necessarias.difference(dados.columns)
if faltantes:
    raise ValueError(f"Colunas esperadas ausentes: {sorted(faltantes)}")

mes = pd.to_datetime(dados["mes_de_referencia"], format="mixed", errors="coerce")
if (mes.isna() & dados["mes_de_referencia"].notna()).any():
    raise ValueError("Há meses de referência em formato inesperado.")

sudeste_2025 = dados.loc[
    mes.dt.year.eq(2025) & dados["uf"].isin(["ES", "MG", "RJ", "SP"])
].copy()
sudeste_2025["mes_de_referencia"] = mes.loc[sudeste_2025.index].dt.to_period("M")
if sudeste_2025["mes_de_referencia"].nunique() != 12:
    raise ValueError("O recorte regional não contém os 12 meses de 2025.")

codigo_produto = pd.to_numeric(sudeste_2025["codigo_anp_do_produto"], errors="coerce")
diesel_s10 = sudeste_2025.loc[codigo_produto.eq(420105001)].copy()
if diesel_s10.empty:
    raise ValueError("O produto 420105001 não foi encontrado no recorte.")

Python 3.13.18
pandas 2.2.3
openpyxl 3.1.5
SHA-256 5e4a123e8013fc9b36bfe54a58a4d8bcb7f5332eef89b7ec8e309cd92e0edc86
Formato recebido: Excel


# 4. Análise exploratória

Checagem inicial de cobertura e descrições do produto.

In [2]:
resumo_carga = pd.DataFrame({
    "Registros": [len(dados_origem), len(sudeste_2025), len(diesel_s10)],
    "Terminais": [
        dados["codigo_anp_do_terminal"].nunique(),
        sudeste_2025["codigo_anp_do_terminal"].nunique(),
        diesel_s10["codigo_anp_do_terminal"].nunique(),
    ],
}, index=["Arquivo de origem", "Sudeste em 2025", "Diesel A S10 no recorte"])

display(resumo_carga)
display(diesel_s10["descricao_do_produto"].value_counts(dropna=False))

,Registros,Terminais
Arquivo de origem,190944,130
Sudeste em 2025,6891,23
Diesel A S10 no recorte,1843,13


,count
descricao_do_produto,
Ã“LEO DIESEL A S10,1352
ÓLEO DIESEL A S10,245
DIESEL S10,189
OLEO DIESEL A S 10,31
OLEO DIESEL S10,15
OLEO DIESEL A S10,11


## 4.1 Granularidade, ausências e cobertura

A chave candidata serve para investigar repetições. A tabela de cobertura conta registros por terminal e mês. Zero na tabela indica ausência de registros no recorte.

In [3]:
chave_candidata = [
    "mes_de_referencia", "codigo_anp_do_terminal", "codigo_anp_do_produto",
    "sentido_da_operacao", "tipo_da_operacao", "modo_de_transporte",
]

ausencias = diesel_s10.isna().sum().rename("ausências")
display(pd.concat([diesel_s10.dtypes.rename("tipo"), ausencias], axis=1))

print("Duplicatas exatas além da primeira ocorrência:", diesel_s10.duplicated().sum())
repeticoes = diesel_s10.duplicated(subset=chave_candidata, keep=False)
print("Registros em chaves candidatas repetidas:", repeticoes.sum())
if repeticoes.any():
    display(
        diesel_s10.loc[repeticoes]
        .groupby(chave_candidata, dropna=False)
        .agg(registros=("volume_m3", "size"), volumes_distintos=("volume_m3", "nunique"))
        .sort_values("registros", ascending=False)
        .head(10)
    )

meses_2025 = pd.period_range("2025-01", "2025-12", freq="M")
cobertura = (
    diesel_s10.groupby(["codigo_anp_do_terminal", "mes_de_referencia"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=meses_2025, fill_value=0)
)
display(cobertura.assign(meses_com_registro=cobertura.gt(0).sum(axis=1)))

,tipo,ausências
mes_de_referencia,period[M],0
codigo_anp_do_terminal,int64,0
nome_do_terminal,object,0
municipio_do_terminal,object,0
uf,object,0
sentido_da_operacao,int64,0
tipo_da_operacao,int64,0
modo_de_transporte,int64,0
codigo_anp_do_produto,int64,0
descricao_do_produto,object,0


Duplicatas exatas além da primeira ocorrência: 6
Registros em chaves candidatas repetidas: 1532


,,,,,,registros,volumes_distintos
mes_de_referencia,codigo_anp_do_terminal,codigo_anp_do_produto,sentido_da_operacao,tipo_da_operacao,modo_de_transporte,,
2025-12,1205529,420105001,2,1,1,179,179
2025-11,1205529,420105001,2,1,1,134,133
2025-10,1205529,420105001,2,1,1,132,131
2025-05,1205529,420105001,2,1,1,119,119
2025-06,1205529,420105001,2,1,1,113,113
2025-04,1205529,420105001,2,1,1,103,103
2025-09,1205529,420105001,2,1,1,93,92
2025-07,1205529,420105001,2,1,1,70,70
2025-03,1205529,420105001,2,1,1,69,69


,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,meses_com_registro
codigo_anp_do_terminal,,,,,,,,,,,,,
1032193,2,2,2,2,2,2,2,2,2,2,2,2,12
1032194,3,1,1,2,4,0,1,1,0,4,1,0,9
1032197,4,4,4,3,6,5,6,6,5,5,5,6,12
1032198,3,3,3,4,3,3,3,3,3,3,3,3,12
1032236,2,3,2,3,3,3,3,2,3,2,3,2,12
1041860,2,2,2,2,2,2,2,2,2,2,1,2,12
1043387,14,11,16,16,18,18,17,16,18,14,16,15,12
1107648,2,2,2,2,2,2,2,2,2,2,2,2,12
1109984,0,0,0,2,1,2,1,1,1,2,1,0,8


## 4.2 Sensibilidade a registros idênticos

A comparação mede o efeito de retirar cópias exatas dentro do recorte. Os dois cenários são calculados sem alterar os dados usados nas outras células.

In [4]:
candidatas_exatas = diesel_s10.duplicated(keep=False)
display(
    diesel_s10.loc[candidatas_exatas, chave_candidata + ["volume_m3"]]
    .sort_values(chave_candidata + ["volume_m3"])
)

comparacao = pd.concat(
    [
        diesel_s10.groupby("sentido_da_operacao")["volume_m3"].sum(min_count=1)
        .rename("volume_com_todas_m3"),
        diesel_s10.drop_duplicates().groupby("sentido_da_operacao")["volume_m3"]
        .sum(min_count=1).rename("volume_sem_copias_exatas_m3"),
    ],
    axis=1,
)
comparacao["diferenca_m3"] = (
    comparacao["volume_com_todas_m3"] - comparacao["volume_sem_copias_exatas_m3"]
)
display(comparacao.rename(index={1: "Recepção", 2: "Entrega"}))

,mes_de_referencia,codigo_anp_do_terminal,codigo_anp_do_produto,sentido_da_operacao,tipo_da_operacao,modo_de_transporte,volume_m3
188836,2025-03,1043387,420105001,1,1,9,0.001
188837,2025-03,1043387,420105001,1,1,9,0.001
188845,2025-05,1043387,420105001,1,1,9,0.001
188847,2025-05,1043387,420105001,1,1,9,0.001
188854,2025-07,1043387,420105001,1,1,9,0.001
188858,2025-07,1043387,420105001,1,1,9,0.001
109619,2025-09,1205529,420105001,2,1,1,57.300
109623,2025-09,1205529,420105001,2,1,1,57.300
110861,2025-10,1205529,420105001,2,1,1,61.790
110876,2025-10,1205529,420105001,2,1,1,61.790


,volume_com_todas_m3,volume_sem_copias_exatas_m3,diferenca_m3
sentido_da_operacao,,,
Recepção,8889369.830,8889369.827,0.003
Entrega,8826921.906,8826745.836,176.070


# 5. Pré-processamento e decisões

A primeira comparação prevista é entre seleção por descrição literal e por código ANP, usando contagem de registros e volume em m³. As outras decisões serão fundamentadas após o diagnóstico. A análise é descritiva e não prevê divisão em treino e teste.

# 6. Respostas e conclusão

Seção a ser elaborada pelo aluno após a análise, com evidências e limites para cada pergunta.

# 7. Checklist de entrega

A revisão final inclui execução em sessão nova no Colab, verificação da versão da fonte, checklist com localizações, referências, declaração de uso de IA e conferência dos links de entrega.